# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/james-kalaw/FlyRank-ML-Intern-Kalaw/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one page.**

The warehouse has one row for each page **for each day**. I add up the days, so I get one row for each page.

I use two time periods:

| Period | Dates | What I take from it |
|---|---|---|
| **Before** | 1–28 Feb 2026 | my 5 clues (features) |
| **Decision day** | 1 Mar 2026 | every clue must already be known on this day |
| **After** | 1–31 Mar 2026 | the answer: did the page decline? |

**Declining** means the page's impressions **per day** in March are more than 20% lower than in February. I use "per day" because February has 28 days and March has 31. If I compared totals, March would look better just because it is longer.

**Which pages I keep:**
- pages from clients with Search Console data starting on or before 1 Feb 2026. Clients joined on different dates, so some don't have a full February.
- pages created before 1 Feb 2026, so they existed for all of February.
- pages with at least 100 impressions in February. Small numbers jump around too much.
- only days where `gsc_data_available IS TRUE`. This flag can be TRUE, FALSE, or empty, and `IS TRUE` keeps only the TRUE days.

I don't use June 2026 (the last month). The card says to keep it for a final test.

**Why this is better than Week 2:** in Week 2, the clues and the answer came from the same 90 days. Now the clues come first and the answer comes after.

The code below connects to the warehouse and counts how many clients have a full February.

**My result (write after Run all):** _How many clients have a full February?_

In [ ]:
# Setup: connect DuckDB to the warehouse (token comes from the Colab Secrets panel, never typed here)
%pip -q install duckdb
import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
# Only the two months I need (a list of paths; hf:// doesn't support {a,b} globs)
FACT_FEB_MAR = (f"read_parquet(['{REL}/fact_content_daily_performance/month=2026-02/*.parquet', "
                f"'{REL}/fact_content_daily_performance/month=2026-03/*.parquet'])")

BEFORE_START, DECISION, AFTER_END = "2026-02-01", "2026-03-01", "2026-03-31"

# How many clients have a full February of Search Console history?
print(con.sql(f"""
    SELECT COUNT(*) AS all_clients,
           COUNT(*) FILTER (WHERE gsc_data_start <= DATE '{BEFORE_START}') AS full_february,
           COUNT(*) FILTER (WHERE gsc_data_start IS NULL) AS no_gsc_start_date
    FROM {DIM_CLIENTS}
""").df().to_string(index=False))

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I sort every column I use into 4 groups.

| Column | Group | Why |
|---|---|---|
| `content_age_days` (from `content_created_date`) | **feature** | the page was made before 1 Mar, so its age is known |
| `word_count` | **feature** | page length (see the limit in Section 4) |
| `search_volume` | **feature** | how many people search the page's keyword (see Section 4) |
| `feb_avg_position` (from `gsc_avg_position`, Feb only) | **feature** | uses February only. Busy days count more than quiet days |
| `feb_impressions` (from `gsc_impressions`, Feb only) | **feature** | uses February only |
| `mar_impressions` (from `gsc_impressions`, Mar only) | **label** | this is what the answer is made from, so using it as a clue is cheating |
| `is_declining` | **label** | the answer |
| `client_hash_id`, `content_hash_id`, `report_date`, `month` | **context** | for joining, grouping, and picking dates. Not clues |
| `gsc_data_available`, `gsc_data_start` | **context** | for keeping only good data |
| `content_updated_date`, `last_optimized_date` | **excluded** | these dates can be after 1 Mar, which is the future |
| `is_deleted`, `is_published` | **excluded** | they show the page today, not on 1 Mar |
| `provider_used`, `model_used` | **excluded** | not page clues (data dictionary) |
| `ga4_*`, `sessions_*`, `ai_*` columns | **excluded for now** | many clients got Google Analytics later, and some values are empty. I start with Search Console only |

**The one thing I leave out on purpose:** anything dated after 1 March, most of all March impressions, because that is the answer itself.

The code writes these groups as lists and checks that no column is in two groups.

In [ ]:
# The contract as code: every field in exactly one bucket
FEATURES = ["content_age_days", "word_count", "search_volume", "feb_avg_position", "feb_impressions"]
LABEL = ["is_declining", "mar_impressions"]            # the answer and its ingredient
CONTEXT = ["client_hash_id", "content_hash_id", "report_date", "month",
           "gsc_data_available", "gsc_data_start"]
EXCLUDED = ["content_updated_date", "last_optimized_date", "is_deleted", "is_published",
            "provider_used", "model_used"]

buckets = {"feature": FEATURES, "label": LABEL, "context": CONTEXT, "excluded": EXCLUDED}
all_fields = [f for group in buckets.values() for f in group]
assert len(all_fields) == len(set(all_fields)), "a field sits in two buckets"
assert not set(FEATURES) & (set(LABEL) | set(EXCLUDED)), "a label or excluded field is a feature"

for name, group in buckets.items():
    print(f"{name:9} ({len(group)}): {', '.join(group)}")
print("\nCheck passed: every field is in exactly one bucket, and no feature is a label or excluded.")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

I check three facts with three queries, using only February and March:

1. **Grain** (what one row stands for): one row is one page on one day, with no duplicates.
2. **Counts and dates:** how many rows and pages each month has, and that the dates go from 1 Feb to 31 Mar.
3. **Good data:** how many rows have `gsc_data_available IS TRUE`, and how many are FALSE or empty.

After that I build my five-feature table, and then I do **the trap**.

In [ ]:
# Query 1 — GRAIN: is (report_date, client, content) unique? Zero rows back = yes.
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_FEB_MAR}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Duplicate page-days found: {len(dupes)}")
print("Grain holds: one row = one page on one day" if dupes.empty else "Grain BROKEN, see rows above")

In [ ]:
# Query 2 — COUNTS + DATE SPAN per month
print(con.sql(f"""
    SELECT month,
           COUNT(*)                         AS rows,
           COUNT(DISTINCT content_hash_id)  AS pages,
           COUNT(DISTINCT client_hash_id)   AS clients,
           MIN(report_date)                 AS first_day,
           MAX(report_date)                 AS last_day,
           COUNT(DISTINCT report_date)      AS days
    FROM {FACT_FEB_MAR}
    GROUP BY month
    ORDER BY month
""").df().to_string(index=False))

In [ ]:
# Query 3 — AVAILABILITY: how many rows survive the IS TRUE filter?
avail = con.sql(f"""
    SELECT month,
           COUNT(*)                                              AS rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)    AS gsc_true,
           COUNT(*) FILTER (WHERE gsc_data_available = FALSE)    AS gsc_false,
           COUNT(*) FILTER (WHERE gsc_data_available IS NULL)    AS gsc_null
    FROM {FACT_FEB_MAR}
    GROUP BY month
    ORDER BY month
""").df()
avail["survive_%"] = (avail["gsc_true"] / avail["rows"] * 100).round(1)
print(avail.to_string(index=False))

### My five-feature table

One SQL query does the heavy work inside DuckDB. Only the small result (one row per page) comes back to pandas.

**When is each feature known?** Every feature must be known on 1 March:
- `content_age_days`: known on 1 Mar, because the page was made earlier.
- `word_count`: known on 1 Mar, because the page already existed. But the value is from today (Section 4).
- `search_volume`: known on 1 Mar, because it describes the keyword. Also a value from today (Section 4).
- `feb_avg_position`: known on 1 Mar, because it uses 1–28 Feb only.
- `feb_impressions`: known on 1 Mar, because it uses 1–28 Feb only.

`mar_impressions` is in the same query only to make the answer. It is never a feature.

In [ ]:
# Build one row per page: Feb features + March outcome
frame = con.sql(f"""
    WITH ok_clients AS (
        SELECT client_hash_id FROM {DIM_CLIENTS}
        WHERE gsc_data_start <= DATE '{BEFORE_START}'
    ),
    per_page AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date <  DATE '{DECISION}') AS feb_impressions,
               SUM(f.gsc_avg_position * f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}')
                 / NULLIF(SUM(f.gsc_impressions) FILTER (WHERE f.report_date < DATE '{DECISION}'), 0)
                                                                                         AS feb_avg_position,
               COUNT(*) FILTER (WHERE f.report_date >= DATE '{DECISION}')                AS mar_days,
               SUM(f.gsc_impressions) FILTER (WHERE f.report_date >= DATE '{DECISION}')  AS mar_impressions
        FROM {FACT_FEB_MAR} f
        JOIN ok_clients USING (client_hash_id)
        WHERE f.gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT p.client_hash_id, p.content_hash_id,
           date_diff('day', d.content_created_date, DATE '{DECISION}') AS content_age_days,
           d.word_count, d.search_volume,
           p.feb_avg_position, p.feb_impressions, p.mar_days, p.mar_impressions
    FROM per_page p
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE d.content_created_date < DATE '{BEFORE_START}'
      AND p.feb_impressions >= 100
""").df()

pages_before = len(frame)
frame = frame[frame["mar_days"] > 0].copy()          # pages with no usable March rows: label unknown
dropped_no_march = pages_before - len(frame)

# Label: March impressions per day more than 20% below February's
feb_per_day = frame["feb_impressions"] / 28
mar_per_day = frame["mar_impressions"] / 31
frame["is_declining"] = (mar_per_day < 0.8 * feb_per_day).astype(int)

print(f"Pages in the frame: {len(frame):,} from {frame['client_hash_id'].nunique()} clients")
print(f"Dropped (no usable March data): {dropped_no_march:,}")
print(f"Base rate: {frame['is_declining'].mean() * 100:.1f}% declining\n")
print("Feature summary (no rows shown):")
print(frame[FEATURES].astype(float).describe().T[["count", "mean", "50%", "min", "max"]].round(1).to_string())
print("\nMissing per feature (%):")
print((frame[FEATURES].isna().mean() * 100).round(1).to_string())

### The trap: cheating on purpose

I train a quick model and score it with **AUC**. AUC is a score where **0.5 = guessing** and **1.0 = perfect**. I test it on clients the model never saw (each client is all in training or all in testing).

1. First, with my 5 honest features.
2. Then I add `mar_impressions`, which is part of the answer. The score should jump close to 1.0. That is cheating, not skill.
3. Then I remove it and keep the honest score.

This is the same lesson as warm-up notebook 02, now on the real warehouse.

In [ ]:
# Quick score: honest features vs. honest + one leaked column
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(frame, frame["is_declining"], groups=frame["client_hash_id"]))
train, test = frame.iloc[train_idx], frame.iloc[test_idx]
assert not set(train["client_hash_id"]) & set(test["client_hash_id"]), "a client is in both sides"

def quick_auc(cols):
    model = HistGradientBoostingClassifier(max_depth=3, random_state=42)
    model.fit(train[cols], train["is_declining"])
    return roc_auc_score(test["is_declining"], model.predict_proba(test[cols])[:, 1])

honest = quick_auc(FEATURES)
leaky = quick_auc(FEATURES + ["mar_impressions"])
print(f"Test: {len(test):,} pages from {test['client_hash_id'].nunique()} unseen clients "
      f"(base rate {test['is_declining'].mean() * 100:.1f}%)")
print(f"AUC, 5 honest features:          {honest:.3f}")
print(f"AUC, + mar_impressions (LEAK):   {leaky:.3f}   <- too good: it contains the answer")

# Delete the leak and keep the honest number
final = frame[["client_hash_id", "content_hash_id"] + FEATURES + ["is_declining"]]
assert "mar_impressions" not in final.columns
print(f"\nLeak removed. Final frame: {final.shape[0]:,} pages x {len(FEATURES)} features. "
      f"Honest AUC kept: {honest:.3f}")

**My result (write after Run all):** _Did the grain hold? How many rows and pages were in Feb and Mar, and what % had good data? How many pages are in my table, and what % are declining? How much did AUC jump with the cheat column?_

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**My main limit: `word_count` and `search_volume` show the page today, not on 1 March.**

They come from `dim_content`, which describes each page as it is now (mid-2026). If someone edited a page after 1 March, its word count today may be different from February. So these two features may carry a little bit of the future. The code counts how many of my pages were changed after 1 March, so I know how big this problem is.

**Other limits:**
- **Clients with short history are left out.** Only clients with a full February are included.
- **Small or missing pages are left out.** Pages need 100+ February impressions and good March data.
- **Only one pair of months.** February → March may not look like other months. I can repeat it on other months later.
- **Search Console only.** No Google Analytics features yet.
- **It shows what happened together, not what caused it.** A feature that goes with decline does not make the page decline.

**My result (write after Run all):** _What % of my pages were changed after 1 March? Is this a small or a big problem?_

In [ ]:
# How big is the snapshot problem? Pages in my frame changed after 1 March
snap = con.sql(f"""
    SELECT COUNT(*) AS pages,
           COUNT(*) FILTER (WHERE content_updated_date > DATE '{DECISION}') AS updated_after_decision,
           COUNT(*) FILTER (WHERE last_optimized_date  > DATE '{DECISION}') AS optimized_after_decision
    FROM {DIM_CONTENT} d
    JOIN (SELECT client_hash_id, content_hash_id FROM final) k USING (client_hash_id, content_hash_id)
""").df()
snap["updated_%"] = (snap["updated_after_decision"] / snap["pages"] * 100).round(1)
snap["optimized_%"] = (snap["optimized_after_decision"] / snap["pages"] * 100).round(1)
print(snap.to_string(index=False))

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.